Consigna:

Un grupo asesor financiero, integrado por expertos en IO de una consultora, ha sido contratado por el directorio de una compañía para ayudarle a maximizar el retorno sobre una inversión de U$S 10.000.000 (diez millones de dólares) en un portafolio constituido por cinco acciones específicas: Apple Inc. (AAPL), Microsoft Corp. (MSFT), Amazon.com Inc. (AMZN), Alphabet Inc. (GOOGL) y Tesla Inc. (TSLA).

El grupo asesor ha analizado los rendimientos esperados anuales de estas acciones: Apple Inc. ofrece un 12%, Microsoft Corp. un 10%, Amazon.com Inc. un 15%, Alphabet Inc. un 9% y Tesla Inc. un 11%. Para garantizar una adecuada diversificación y minimizar riesgos, el grupo asesor recomienda invertir al menos U$S 1.000.000 en Apple Inc., U$S 1.500.000 en Microsoft Corp., U$S 2.000.000 en Amazon.com Inc., U$S 500.000 en Alphabet Inc. y U$S 1.000.000 en Tesla Inc.

Además, el grupo asesor establece que la inversión conjunta en acciones de Apple Inc. (AAPL) y Microsoft Corp. (MSFT) no debe exceder el doble de la inversión en las acciones de Amazon.com Inc. (AMZN) y Alphabet Inc. (GOOGL), Por último, sugiere que la inversión en Apple Inc. (AAPL) no debe exceder la inversión en Microsoft Corp. (MSFT) en más de U$S 1.000.000

#### 1- Importación de librerías

In [310]:
import pandas as pd
import pulp as pu
from pulp import LpMaximize
from pulp import LpContinuous
from pulp import LpStatus
from pulp import value

#### 2- Creación del problema de programación lineal denominado "IOL"

In [311]:
problema = pu.LpProblem("IOL", LpMaximize)

#### 3- Definición de las variables de decisión

In [312]:
x1 = pu.LpVariable("x1", lowBound=0, cat=LpContinuous)
x2 = pu.LpVariable("x2", lowBound=0, cat=LpContinuous)
x3 = pu.LpVariable("x3", lowBound=0, cat=LpContinuous)
x4 = pu.LpVariable("x4", lowBound=0, cat=LpContinuous)
x5 = pu.LpVariable("x5", lowBound=0, cat=LpContinuous)

#### 4- Asignación de nuestras variables de decisión al problema creado inicialmente

#### acción	   rendimiento

##### APPL -> 0,12

##### MSFT	-> 0,1

##### AMZN0 -> 0,15

##### GOOGL	-> 0,09

##### TSLA	->   0,11


In [313]:
problema+= 0.12*x1+0.1*x2+0.15*x3+0.09*x4+0.11*x5, "Utilidad"

#### 5- Definimos las restricciones y las asignamos a nuestro problema

In [314]:
problema+= x1+x2+x3+x4+x5 == 10000000, "R1"
problema+= x1  >= 1000000 , "R2"
problema+= x2   >= 1500000 , "R3"
problema+= x3  >= 2000000 , "R4"
problema+= x4  >= 500000 , "R5"
problema+= x5  >= 1000000 , "R6"
problema+= x1 + x2 <= (x3 + x4)*2 , "R7"
problema+= x1 <= x2 + 1000000 , "R8"

#### 6- Resolución del problema

In [315]:
problema.solve()

1

In [316]:
LpStatus[problema.status]

'Optimal'

In [317]:
print("Los valores óptimos de las variables de decisión son:")
print("Monto total a invertir en APPL : $", round(x1.varValue))
print("Monto total a invertir en MSFT : $", round(x2.varValue)) 
print("Monto total a invertir en AMZN : $", round(x3.varValue))
print("Monto total a invertir en GOOGL : $", round(x4.varValue))
print("Monto total a invertir en TSLA : $", round(x5.varValue))

print("El valor de la función objetivo evaluado en el punto óptimo es: ")
print("Utilidad máxima: $",round(value(problema.objective)))

monto_disponible = 10000000
print("El rendimiento anual de la inversión es de: ", ((value(problema.objective)/monto_disponible)*100), "%" )

Los valores óptimos de las variables de decisión son:
Monto total a invertir en APPL : $ 1000000
Monto total a invertir en MSFT : $ 1500000
Monto total a invertir en AMZN : $ 6000000
Monto total a invertir en GOOGL : $ 500000
Monto total a invertir en TSLA : $ 1000000
El valor de la función objetivo evaluado en el punto óptimo es: 
Utilidad máxima: $ 1325000
El rendimiento anual de la inversión es de:  13.25 %


### Quality checks para validación de restricciones

In [318]:
total_invertido = x1.varValue+ x2.varValue +x3.varValue + x4.varValue + x5.varValue

if total_invertido <= 10000000:
    print("Cumple restricción 1")

else:
        print("Cumple restricción 1")

if x1.varValue >= 1000000:
    print("Cumple restricción 2")

else:
        print("No cumple restricción 2")

if x2.varValue >= 1500000:
    print("Cumple restricción 3")

else:
        print("No cumple restricción 3")

if x3.varValue >= 2000000:
    print("Cumple restricción 4")

else:
        print("No cumple restricción 4")

if x4.varValue >= 500000:
    print("Cumple restricción 5")

else:
        print("No cumple restricción 5")

if x5.varValue >= 1000000:
    print("Cumple restricción 6")

else:
        print("No cumple restricción 6")

if (x1.varValue+x2.varValue) <= (x3.varValue+x4.varValue)*2:
    print("Cumple restricción 7")

else:
        print("No cumple restricción 7")

if x1.varValue <= (x2.varValue+1000000):
    print("Cumple restricción 8")

else:
        print("No cumple restricción 8")

Cumple restricción 1
Cumple restricción 2
Cumple restricción 3
Cumple restricción 4
Cumple restricción 5
Cumple restricción 6
Cumple restricción 7
Cumple restricción 8


#### 7- Análisis de sensibilidad

In [319]:
sensibilidad = [{"Restricción":name, "Precio sombra":c.pi, "Holgura": c.slack} for name, c in problema.constraints.items()]
sensibilidad_df = pd.DataFrame(sensibilidad)
sensibilidad_df

,Restricción,Precio sombra,Holgura
0,R1,0.15,-0.0
1,R2,-0.03,-0.0
2,R3,-0.05,-0.0
3,R4,-0.00,-4000000.0
4,R5,-0.06,-0.0
5,R6,-0.04,-0.0
6,R7,-0.00,10500000.0
7,R8,-0.00,1500000.0


Dado que el precio sombra de mi restricción 1 (límite de monto total a invertir) es del 0.15 (15%), ello me indica que por cada $1 que le añada al monto de mi cartera, mi retorno será de 0.15. Por lo tanto, 15% es el máximo costo permitido de un fondo adicional para el portfolio, lo cual vuelve inconveniente la posibilidad de tomar un préstamo de un  U$S 1.000.000 con una tasa anual de 17%.

En caso de que las acciones de GOOGL aumenten su rentabilidad a un 12%, la composición de la cartera no se alterará, dado que el objetivo coeficiente es de 0.09 y es permisible aumentar un 0.06 (0.09 + 0.06 = 0.15), por lo tanto el retorno de la cartera incrementará en un 1.1% (+U$S15,000) 